<!-- DRAFT: placeholder prose; the author will rewrite this. -->
# 02 · Data model
**You'll learn:** how to keep documents, edges, a named graph, a BM25 view and a vector index in one database, and how to build a graph from structure already in your data (links, codes, versions, names).

One database holds documents (articles with their versions and embeddings, tickets, forms, states, lenders, lessors, stores, concepts) and edges. The **structural** edges are parsed from the knowledge base itself (links, form codes, version chains, names). The **semantic** edges come from concepts Claude extracted from each article (cached in `data/kb/semantic/`). Nothing in the graph is written by hand.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, '..'); sys.path.insert(0, '../scripts')
from cookbook.build import build, SHOW_KB_PROBLEMS
from cookbook.db import get_db, run
counts = build()
db = get_db()

W1118 20:17:24.464000 34950 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

database northwind_kb: built in 9.5s
  articles        68  document
  tickets        200  document
  forms           15  document
  states           5  document
  lenders         12  document
  lessors          6  document
  stores          25  document
  concepts       365  document
  links_to       138  edge
  supersedes      17  edge
  uses_form       24  edge
  replaced_by      2  edge
  applies_in       7  edge
  about_lender     7  edge
  about_lessor    18  edge
  cites          200  edge
  mentions       689  edge
  relates        453  edge
  graph kb_graph, view kb_view, vector index articles_embedding_cosine (nLists=4)
  semantic layer: 0 extracted relations skipped (they named a concept the article didn't list)



<!-- DRAFT: placeholder prose; the author will rewrite this. -->
A graph query a support team can use: current articles that still point readers at a retired form.

In [2]:
run(db, SHOW_KB_PROBLEMS)

AQL> FOR a IN articles
       FILTER a.effective_to == null                      // current versions only
       FOR f IN 1 OUTBOUND a uses_form
         FILTER f.status == "retired"
         FILTER f.replaced_by NOT IN a.codes              // cites the old form without its replacement
         RETURN {article: a._key, cites: f._key, replaced_by: f.replaced_by}


[{'article': 'selling_for_someone_else__2022',
  'cites': 'NW-POA-1',
  'replaced_by': 'NW-POA-2'},
 {'article': 'odometer_problems__2022',
  'cites': 'NW-ODO-1',
  'replaced_by': 'NW-ODO-2'}]

<!-- DRAFT: placeholder prose; the author will rewrite this. -->
Everything one article connects to, in both layers.

In [3]:
run(db, """
FOR v, e IN 1..1 ANY "articles/inherited_car__2026" GRAPH "kb_graph"
  OPTIONS { edgeCollections: ["links_to", "uses_form", "mentions"] }
  COLLECT edge = PARSE_IDENTIFIER(e._id).collection INTO targets = v._key
  RETURN { edge, targets }
""")

AQL> FOR v, e IN 1..1 ANY "articles/inherited_car__2026" GRAPH "kb_graph"
       OPTIONS { edgeCollections: ["links_to", "uses_form", "mentions"] }
       COLLECT edge = PARSE_IDENTIFIER(e._id).collection INTO targets = v._key
       RETURN { edge, targets }


[{'edge': 'links_to',
  'targets': ['selling_for_someone_else__2022',
   'co_owners__2026',
   'selling_with_loan__2022',
   'lost_title__2022']},
 {'edge': 'mentions',
  'targets': ['title',
   'small_estate',
   'probate_court',
   'power_of_attorney',
   'outstanding_loan',
   'multiple_heir',
   'lost_title',
   'lienholder',
   'letters_testamentary',
   'inherited_car',
   'heir_and_estate_affidavit',
   'heir',
   'executor',
   'death_certificate']},
 {'edge': 'uses_form', 'targets': ['NW-EST-1']}]